# PathSnap — Bayesian hyperparameter optimisation of HistGradientBoosting

**Why this model:** HistGradientBoosting (Model 2) had the best student_test top-1 of the models compared (89.6%), is fast to train, and has several hyperparameters that genuinely trade bias against variance — so it is the most promising one to tune.

**Why Bayesian (TPE):** each trial costs 0.5–6 minutes, so an exhaustive grid is unaffordable. Optuna's Tree-structured Parzen Estimator builds a probability model of which settings scored well and samples the next trial where improvement is most likely, so a few dozen trials explore a six-dimensional space far better than a grid of the same size.

**Objective:** maximise **observation-level top-1 accuracy** (mean over group-aware CV folds) — the primary metric used to compare all models. Log loss is recorded for every trial as a second check, because the HMM uses the probabilities.

**Validation design (no leakage):**
* Search: **3-fold GroupKFold by trip** on the 76 usable train trips, with **pruning** — a trial whose first fold is clearly below the median of earlier trials is stopped early.
* Final comparison: the **same 5-fold GroupKFold** used in every model notebook, tuned vs untuned side by side.
* Student_test is used **once**, after the configuration is fixed, to report tuned vs untuned. It never influences the search. The evaluator-test partition is not used anywhere.

**Before running on Colab:** `pathsnap_features.zip` in `MyDrive/PathSnap/`. Runtime is set by `N_TRIALS` and `TIMEOUT_MIN` below (default: 40 trials or 45 minutes, whichever comes first).

In [ ]:
# ---- Paths: Colab (Drive zip) or local run --------------------------------------------
import os, sys, zipfile, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = Path("/content/drive/MyDrive/PathSnap/pathsnap_features.zip")
    DATA = Path("/content/pathsnap_features")
    RESULTS = Path("/content/drive/MyDrive/PathSnap/results")
    if not (DATA / "data_features").exists():
        with zipfile.ZipFile(ZIP) as z:
            z.extractall(DATA)
else:
    DATA = Path(os.environ.get("PATHSNAP_DATA", "../.."))
    RESULTS = Path(os.environ.get("PATHSNAP_RESULTS", "results"))

try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna

OUT = RESULTS / "hgb_tuning"
FEAT_DIR = DATA / "data_features"
FLAGS = DATA / "trip_flags.csv" if (DATA / "trip_flags.csv").exists() else DATA / "data_clean" / "trip_flags.csv"
OUT.mkdir(parents=True, exist_ok=True)
print("features:", FEAT_DIR, "| output:", OUT, "| optuna", optuna.__version__)

In [ ]:
import json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss, f1_score

SEED = 42
np.random.seed(SEED)
FEATURES = json.loads((FEAT_DIR / "feature_params.json").read_text())["FEATURES"]

N_TRIALS = int(os.environ.get("PATHSNAP_TRIALS", 40))
TIMEOUT_MIN = float(os.environ.get("PATHSNAP_TIMEOUT_MIN", 45))
SEARCH_FOLDS, FINAL_FOLDS = 3, 5

# Untuned baseline = exactly the Model 2 notebook's configuration
BASELINE = {"learning_rate": 0.1, "max_iter": 300, "max_leaf_nodes": 31, "min_samples_leaf": 20,
            "l2_regularization": 0.0, "max_features": 1.0}
FIXED = {"max_bins": 255, "early_stopping": False, "random_state": SEED}

def make_model(params):
    return HistGradientBoostingClassifier(**params, **FIXED)

print(f"sklearn {sklearn.__version__} | {len(FEATURES)} features | up to {N_TRIALS} trials / {TIMEOUT_MIN:.0f} min")

## 1. Load data
Same as the model notebooks: usable train trips only, `label = -1` rows dropped, per-observation reference kept so an observation with no correct candidate counts as wrong.

In [ ]:
flags = pd.read_csv(FLAGS, dtype={"trip_id": str})

def trips_of(part):
    t = flags[flags.partition == part]
    if part == "train":
        t = t[t.use_for_tuning.astype(bool)]
    return sorted(t.trip_id)

def load_partition(part):
    rows, pts = [], []
    for tid in trips_of(part):
        f = pd.read_csv(FEAT_DIR / part / f"{tid}.csv.gz", dtype={"trip_id": str})
        f["label"] = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}.csv.gz").label.values
        rows.append(f[f.label >= 0])
        p = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}_points.csv.gz")
        p.insert(0, "trip_id", tid)
        pts.append(p[p.labelled])
    return pd.concat(rows, ignore_index=True), pd.concat(pts, ignore_index=True)

train, train_pts = load_partition("train")
test, test_pts = load_partition("student_test")
print(f"train {train.trip_id.nunique()} trips, {len(train):,} rows | student_test {test.trip_id.nunique()} trips, {len(test):,} rows")

def top1(df, score, pts):
    d = df[["trip_id", "idx", "label"]].assign(s=score)
    hits = int(d.loc[d.groupby(["trip_id", "idx"]).s.idxmax()].label.sum())
    return hits / len(pts[pts.trip_id.isin(df.trip_id.unique())])

def fold_scores(params, n_folds, trial=None):
    # group-aware CV; reports the running mean to Optuna after each fold so weak trials can be pruned
    tops, losses, fit_s = [], [], 0.0
    for k, (tr, va) in enumerate(GroupKFold(n_splits=n_folds).split(train, groups=train.trip_id)):
        a, b = train.iloc[tr], train.iloc[va]
        t0 = time.time(); m = make_model(params).fit(a[FEATURES], a.label); fit_s += time.time() - t0
        p = m.predict_proba(b[FEATURES])[:, 1]
        tops.append(top1(b, p, train_pts)); losses.append(log_loss(b.label, np.clip(p, 1e-7, 1 - 1e-7)))
        if trial is not None:
            trial.report(float(np.mean(tops)), k)
            if trial.should_prune():
                trial.set_user_attr("folds_done", k + 1)
                raise optuna.TrialPruned()
    return tops, losses, fit_s

## 2. Search space

| Hyperparameter | Range | Scale | Why |
| --- | --- | --- | --- |
| `learning_rate` | 0.02 – 0.3 | log | step size per tree; smaller needs more trees but generalises better |
| `max_iter` | 100 – 600 (step 50) | linear | number of boosting rounds; works with the learning rate |
| `max_leaf_nodes` | 15 – 127 | log | tree complexity: how many feature interactions one tree can express |
| `min_samples_leaf` | 10 – 300 | log | smallest leaf; larger values stop trees fitting single noisy GPS points |
| `l2_regularization` | 1e-4 – 10 | log | shrinks leaf values; guards against overconfident probabilities |
| `max_features` | 0.5 – 1.0 | linear | share of the 24 features tried per split; decorrelates trees |

Upper limits keep the slowest trial to about 6 minutes on Colab. Trial 0 is the untuned baseline, so it is scored by exactly the same procedure as every candidate.

In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 600, step=50),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 127, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 300, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10.0, log=True),
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
    }
    t0 = time.time()
    tops, losses, fit_s = fold_scores(params, SEARCH_FOLDS, trial)
    trial.set_user_attr("cv_log_loss", float(np.mean(losses)))
    trial.set_user_attr("cv_top1_std", float(np.std(tops)))
    trial.set_user_attr("trial_s", time.time() - t0)
    trial.set_user_attr("folds_done", SEARCH_FOLDS)
    return float(np.mean(tops))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", study_name="hgb_top1",
                            sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=8),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0))
study.enqueue_trial({**BASELINE, "l2_regularization": 1e-4})   # baseline (l2=0 sits at the range floor)

def progress(study, trial):
    state = "pruned" if trial.state == optuna.trial.TrialState.PRUNED else f"top-1 {trial.value:.4f}"
    print(f"trial {trial.number:2d}: {state:16s} | best so far {study.best_value:.4f} (trial {study.best_trial.number})")

t_search = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_MIN * 60, callbacks=[progress])
search_min = (time.time() - t_search) / 60
done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
print(f"\n{len(study.trials)} trials ({len(done)} complete, {len(study.trials) - len(done)} pruned) in {search_min:.1f} min")
print("best 3-fold top-1:", round(study.best_value, 4), "| baseline (trial 0):", round(study.trials[0].value, 4))
BEST = dict(study.best_params)
BEST

## 3. Search results

In [ ]:
trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state", "duration"))
trials.to_csv(OUT / "optuna_trials.csv", index=False)

comp = trials[trials.state == "COMPLETE"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(comp.number, comp.value, color="#2a78d6", s=25, label="completed trial")
axes[0].plot(comp.number, comp.value.cummax(), color="#eb6834", lw=2, label="best so far")
axes[0].axhline(study.trials[0].value, color="#9c9b96", ls="--", lw=1, label="untuned baseline")
axes[0].set_xlabel("trial"); axes[0].set_ylabel("3-fold CV top-1"); axes[0].legend(); axes[0].set_title("Optimisation history")

try:
    imp = optuna.importance.get_param_importances(study)
    axes[1].barh(list(imp.keys())[::-1], list(imp.values())[::-1], color="#2a78d6")
    axes[1].set_title("Hyperparameter importance (fANOVA)")
except Exception as e:
    axes[1].text(0.5, 0.5, f"importance unavailable: {e}", ha="center"); axes[1].axis("off")
fig.tight_layout(); fig.savefig(OUT / "tuning_history_importance.png", dpi=150); plt.show()

cols = ["number", "value", "user_attrs_cv_log_loss", "params_learning_rate", "params_max_iter", "params_max_leaf_nodes",
        "params_min_samples_leaf", "params_l2_regularization", "params_max_features", "user_attrs_trial_s"]
comp.sort_values("value", ascending=False)[cols].head(8).round(4)

## 4. Tuned vs untuned — same 5-fold GroupKFold as every model notebook
The search used 3 folds; the comparison uses the standard 5 folds so the numbers line up with the other models. **The decision to adopt the tuned configuration is made here, on cross-validation**, not on student_test.

In [ ]:
rows = []
for name, params in [("untuned", BASELINE), ("tuned", BEST)]:
    tops, losses, fit_s = fold_scores(params, FINAL_FOLDS)
    for k, (t, l) in enumerate(zip(tops, losses)):
        rows.append({"config": name, "fold": k, "top1": t, "log_loss": l})
    print(f"{name:8s}: 5-fold top-1 {np.mean(tops):.4f} ± {np.std(tops):.4f} | log loss {np.mean(losses):.4f} | fit {fit_s:.0f}s total")
cv5 = pd.DataFrame(rows)
cv5_summary = cv5.groupby("config")[["top1", "log_loss"]].agg(["mean", "std"])
ADOPT = cv5[cv5.config == "tuned"].top1.mean() > cv5[cv5.config == "untuned"].top1.mean()
print("\nDecision (on CV only):", "ADOPT the tuned configuration" if ADOPT else "KEEP the untuned configuration")
cv5.pivot(index="fold", columns="config", values="top1").round(4)

## 5. Student_test — reported once, after the decision

In [ ]:
final = {}
for name, params in [("untuned", BASELINE), ("tuned", BEST)]:
    t0 = time.time(); m = make_model(params).fit(train[FEATURES], train.label); fit_s = time.time() - t0
    t0 = time.time(); p = m.predict_proba(test[FEATURES])[:, 1]; pred_s = time.time() - t0
    final[name] = {"model": m, "p": p, "top1": top1(test, p, test_pts),
                   "log_loss": log_loss(test.label, np.clip(p, 1e-7, 1 - 1e-7)),
                   "f1": f1_score(test.label, (p >= 0.5).astype(int)), "fit_s": fit_s, "predict_s": pred_s}
comparison = pd.DataFrame({
    "5-fold CV top-1": cv5.groupby("config").top1.mean(),
    "5-fold CV top-1 std": cv5.groupby("config").top1.std(),
    "5-fold CV log loss": cv5.groupby("config").log_loss.mean(),
    "student_test top-1": {k: v["top1"] for k, v in final.items()},
    "student_test log loss": {k: v["log_loss"] for k, v in final.items()},
    "student_test F1": {k: v["f1"] for k, v in final.items()},
    "final fit s": {k: v["fit_s"] for k, v in final.items()},
}).loc[["untuned", "tuned"]]
comparison.round(4)

## 6. Save
* `best_params.json` — tuned configuration, the adopt/keep decision and its evidence. The HMM notebook and the API build script read this file.
* `hgb_tuned_model.joblib` — the tuned model refitted on all usable train trips.
* `optuna_trials.csv` — every trial; `experiment_log.json` — full record including compute cost.

In [ ]:
chosen = "tuned" if ADOPT else "untuned"
(OUT / "best_params.json").write_text(json.dumps({
    "adopted": bool(ADOPT), "chosen": chosen, "params": BEST if ADOPT else BASELINE,
    "tuned_params": BEST, "baseline_params": BASELINE, "fixed": FIXED,
    "objective": "mean observation-level top-1, 3-fold GroupKFold by trip (train only)",
    "decision_rule": "adopt if 5-fold GroupKFold CV top-1 improves over the untuned baseline"}, indent=2))
joblib.dump(final["tuned"]["model"], OUT / "hgb_tuned_model.joblib")
(OUT / "experiment_log.json").write_text(json.dumps({
    "search": {"sampler": "TPE (seed 42, 8 random start-up trials)", "pruner": "Median (after fold 1, from trial 5)",
               "folds": SEARCH_FOLDS, "n_trials": len(study.trials), "complete": len(done),
               "pruned": len(study.trials) - len(done), "minutes": search_min,
               "best_value": study.best_value, "baseline_value": study.trials[0].value},
    "cv5": cv5.to_dict(orient="records"),
    "student_test": {k: {kk: vv for kk, vv in v.items() if kk not in ("model", "p")} for k, v in final.items()},
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "optuna": optuna.__version__},
}, indent=2, default=float))
test[["trip_id", "idx", "arc_id", "label"]].assign(p_untuned=final["untuned"]["p"], p_tuned=final["tuned"]["p"]).to_csv(
    OUT / "student_test_predictions.csv.gz", index=False, float_format="%.6f")
print("saved to", OUT, "| chosen:", chosen)